# 09 — Stacking Ensemble (Stage 11)
Base learners: the tuned Random Forest, XGBoost and CatBoost. Meta-learner: class-balanced Logistic Regression trained on *out-of-fold* predicted probabilities (inner 5-fold CV), which prevents leakage. GridSearchCV over the meta-learner's C.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)
FIG = ROOT / 'figures'; REP = ROOT / 'reports'

In [2]:
summary = pd.read_csv(REP / 'tuning' / 'tuning_summary.csv').set_index('model')
row = summary.loc['stacking']
print('Search             :', row['search'], f"({row['n_candidates']} candidates)")
print('CV Macro F1        :', f"{row['cv_macro_f1_mean']:.4f} ± {row['cv_macro_f1_std']:.4f}")
print('Train Macro F1     :', f"{row['train_macro_f1_mean']:.4f}")
print('Train - CV gap     :', f"{row['train_minus_cv_gap']:+.4f}")
print('Best parameters    :', row['best_params'])

Search             : GridSearchCV (3 candidates)
CV Macro F1        : 0.7881 ± 0.0101
Train Macro F1     : 0.8103
Train - CV gap     : +0.0222
Best parameters    : {'model__final_estimator__C': 1.0}


Fold-wise Macro F1 (same 5 stratified folds for every model):

In [3]:
folds = pd.read_csv(REP / 'tuning' / 'cv_fold_scores.csv', index_col=0)
folds.loc[[i for i in folds.index if i.startswith('Stacking')]]

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
Stacking,0.7778,0.8016,0.7778,0.7984,0.7848,0.7881,0.0101


Top 10 candidates of the search, ranked by mean CV Macro F1:

In [4]:
res = pd.read_csv(REP / 'tuning' / 'stacking_search_results.csv')
cols = [c for c in res.columns if c.startswith('param_')] + ['mean_test_score', 'std_test_score', 'mean_train_score', 'mean_fit_time']
res.sort_values('rank_test_score')[cols].head(10).round(4)

,param_model__final_estimator__C,mean_test_score,std_test_score,mean_train_score,mean_fit_time
1,1.0,0.7881,0.0101,0.8103,62.4349
0,0.1,0.7876,0.0113,0.8209,62.1569
2,10.0,0.7855,0.0119,0.8080,61.8699


## Initial vs tuned, all ensembles

In [5]:
pd.read_csv(REP / 'tuning/initial_vs_tuned.csv')

,model,initial_cv_macro_f1,initial_cv_std,tuned_cv_macro_f1,tuned_cv_std,improvement
0,Random Forest,0.7842,0.0094,0.7909,0.0129,0.0067
1,XGBoost,0.7620,0.0251,0.7956,0.0294,0.0336
2,CatBoost,0.7551,0.0104,0.7766,0.0304,0.0215


## Complete tuning summary (Stages 11–12)

In [6]:
pd.read_csv(REP / 'tuning/tuning_summary.csv')

,model,label,search,n_candidates,best_params,cv_macro_f1_mean,cv_macro_f1_std,train_macro_f1_mean,seconds,train_minus_cv_gap
0,dummy,"Baseline (Dummy, most frequent)",none (fixed strategy),1,{},0.196874,0.000000,0.196874,0.1,0.000000
1,logistic_regression,Baseline (Logistic Regression),GridSearchCV,10,"{'model__C': 10.0, 'model__class_weight': None}",0.708359,0.041934,0.714312,3.4,0.005953
2,random_forest,Random Forest,RandomizedSearchCV,30,"{'model__class_weight': 'balanced', 'model__ma...",0.790863,0.012894,0.815664,270.5,0.024802
3,xgboost,XGBoost,RandomizedSearchCV,30,"{'model__class_weight': 'balanced', 'model__co...",0.795566,0.029407,0.993593,84.6,0.198027
4,catboost,CatBoost,RandomizedSearchCV,20,"{'model__auto_class_weights': 'SqrtBalanced', ...",0.776616,0.030356,1.000000,298.9,0.223384
5,stacking,Stacking,GridSearchCV,3,{'model__final_estimator__C': 1.0},0.788103,0.010121,0.810305,565.6,0.022202


In [7]:
pd.read_csv(REP / 'tuning/cv_fold_scores.csv', index_col=0)

,fold_1,fold_2,fold_3,fold_4,fold_5,mean,std
Random Forest (initial),0.7720,0.7779,0.7831,0.7995,0.7883,0.7842,0.0094
XGBoost (initial),0.7708,0.7153,0.7676,0.7911,0.7651,0.7620,0.0251
CatBoost (initial),0.7636,0.7534,0.7413,0.7698,0.7476,0.7551,0.0104
"Baseline (Dummy, most frequent)",0.1969,0.1969,0.1969,0.1969,0.1969,0.1969,0.0000
Baseline (Logistic Regression),0.7867,0.6784,0.7176,0.6818,0.6772,0.7084,0.0419
Random Forest,0.7870,0.7801,0.7980,0.8122,0.7770,0.7909,0.0129
XGBoost,0.8210,0.7978,0.7823,0.8295,0.7472,0.7956,0.0294
CatBoost,0.8062,0.7423,0.7813,0.8122,0.7411,0.7766,0.0304
Stacking,0.7778,0.8016,0.7778,0.7984,0.7848,0.7881,0.0101
